In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

import torchvision.models as models

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
BATCH_SIZE = 256

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse',
           'ship', 'truck')

from torch.utils.data import random_split

full_train_dataset = datasets.CIFAR10(root='./data', train=True,
                                      download=True, transform=transform_train)

test_dataset = datasets.CIFAR10(root='./data', train=False,
                                      download=True, transform=transform_test)

train_dataset, val_dataset = random_split(full_train_dataset, [0.8, 0.2])


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
finder_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
class ExpandingHead(nn.Module):
    def __init__(self, num_classes=10):
        super(ExpandingHead, self).__init__()
        
        backbone = models.vgg19_bn(weights=None)
        self.backbone = backbone.features
        self.fc0 = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.fc0(x)
        return x
    
    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

In [5]:
model = ExpandingHead()

In [6]:
model( torch.randn(2, 3, 32, 32))

tensor([[ 0.1955, -0.8900, -0.7511,  0.1929,  0.4699,  0.1312, -0.9956,  0.5894,
          0.7413,  0.1472],
        [-0.1865, -0.5173, -0.2077,  0.8430,  0.3553,  0.0500, -0.0854,  0.3543,
          0.3925,  0.3048]], grad_fn=<AddmmBackward0>)

In [7]:
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

Converting layer 'fc0' to ScaledNet...


In [8]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.01},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.01,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 4000,
    "weight_decay": 4e-3,
    "diff_threshold": 0.05
}

In [9]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="cifar10-backbone",
)
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/vanya9090/cifar10-backbone/820466b624304b46ab1425c2b6098fb8



In [10]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance):
    return optim.SGD(
        model_instance.parameters(), 
        lr=hyperparams['learning_rate'], 
        weight_decay=hyperparams['weight_decay']
    )

trained_model = train_ssonn(
    model, 
    train_loader, 
    val_loader, 
    finder_loader,
    criterion, 
    optimizer_factory, 
    hyperparams, 
    device,
    experiment,
    True
)

--- Starting SSONN Training ---


Epoch 1/256 | T-Loss: 2.2103 | V-Loss: 2.0908 | Met: 0.2293 | Params: 5130/5130


Epoch 2/256 | T-Loss: 1.9965 | V-Loss: 1.9300 | Met: 0.2824 | Params: 5130/5130


Epoch 3/256 | T-Loss: 1.8679 | V-Loss: 1.8382 | Met: 0.3189 | Params: 5130/5130


Epoch 4/256 | T-Loss: 1.7900 | V-Loss: 1.7639 | Met: 0.3446 | Params: 5130/5130


Epoch 5/256 | T-Loss: 1.7237 | V-Loss: 1.7137 | Met: 0.3712 | Params: 5130/5130


... (247 outputs left out)

Epoch 253/256 | T-Loss: 0.4350 | V-Loss: 0.7194 | Met: 0.7624 | Params: 170265124/170465130


Epoch 254/256 | T-Loss: 0.4314 | V-Loss: 0.7122 | Met: 0.7621 | Params: 170265124/170465130


Epoch 255/256 | T-Loss: 0.4295 | V-Loss: 0.7313 | Met: 0.7587 | Params: 170265124/170465130


Epoch 256/256 | T-Loss: 0.4307 | V-Loss: 0.7268 | Met: 0.7587 | Params: 170265124/170465130
--- Training Finished ---


In [28]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : trash
COMET INFO:     url                   : https://www.comet.com/vanya9090/cifar10-backbone/179f1e339ae4481bbeba9bc311a8b556
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     lr                : 0.001
COMET INFO:     params amount     : 5130
COMET INFO:     sparsity          : 0.0
COMET INFO:     total_params      : 5130
COMET INFO:     train loss [69]   : (0.37638602192234843, 1.732539752486405)
COMET INFO:     val accuracy [69] : (0.3082, 0.8396)
COMET INFO:     val loss [69]     : (0.518491180241108, 2.0290901213884354)
COMET INFO:   Others:
COMET INFO:     Name            : trash
COMET INFO:     hasNestedParams : True
COMET INFO:   Par